In [11]:
from pathlib import Path
import re

In [2]:
def read_text_file(file_path: Path) -> tuple[str, str]:
    """
    Read a text file using UTF-8 or ISO-8859-1 encoding.

    Args:
        file_path: Path to the text file.

    Returns:
        A tuple containing the file content and the encoding used.
    """
    try:
        file_text_content = file_path.read_text(encoding="utf-8")
        encoding_used = "UTF-8"

    except UnicodeDecodeError:
        file_text_content = file_path.read_text(encoding="ISO-8859-1")
        encoding_used = "ISO-8859-1"

    return file_text_content, encoding_used

In [ ]:
file_path = Path(
    r"C:\Users\Bruker\INF_prosjekter\shared_data\d2_data\2026\library_cafe\POS_01\register_sales.log"
)

file_text_content, encoding_used = read_text_file(file_path)

print(file_text_content)
print(f"Encoding used: {encoding_used}")

[2026-10-15 07:00:38] [POS_01] item=Croissant price=3.98 EUR (SUCCESS)
[2026-10-15 07:02:55] [POS_01] item=Sandwich price=7.70 USD (SUCCESS)
[2026-10-15 07:05:45] [POS_01] item=Sandwich price=7.43 USD (SUCCESS)
[2026-10-15 07:07:17] [POS_01] item=Cappuccino price=3.72 USD (SUCCESS)
[2026-10-15 07:08:34] [POS_01] item=Muffin price=3.45 EUR (SUCCESS)
[2026-10-15 07:10:12] [POS_01] item=Muffin price=3.37 NOK (SUCCESS)
[2026-10-15 07:11:39] [POS_01] item=Muffin price=3.20 EUR (REFUND)
=== NETWORK SYNC OK ===
[2026-10-15 07:13:17] [POS_01] item=Croissant price=3.88 USD (CANCELLED)
[2026-10-15 07:16:17] [POS_01] item=Espresso price=3.39 NOK (CANCELLED)
[2026-10-15 07:17:01] [POS_01] item=Espresso price=3.31 EUR (REFUND)
[2026-10-15 07:18:30] [POS_01] item=Croissant price=3.31 USD (SUCCESS)
[2026-10-15 07:19:18] [POS_01] item=Cappuccino price=4.48 NOK (CANCELLED)
[2026-10-15 07:19:45] [POS_01] item=Espresso price=2.87 USD (SUCCESS)
[2026-10-15 07:20:14] [POS_01] item=Muffin price=3.99 NOK (SU

In [12]:
log_pattern = re.compile(
    r"^\[(?P<timestamp>\d{4}-\d{2}-\d{2} \d{2}:\d{2}:\d{2})\] "
    r"\[(?P<register_id>[^\]\s]+)\] "
    r"item=(?P<item>.+?) "
    r"price=(?P<price>\d+(?:\.\d+)?) "
    r"(?P<currency>[A-Z]{3}) "
    r"\((?P<status>SUCCESS|REFUND|CANCELLED)\)$",
    re.MULTILINE
)

In [14]:
def parse_log_file(file_text_content: str) -> list[dict]:
    """
    Parse valid transaction lines from log text.

    Args:
        file_text_content: The complete log file content.

    Returns:
        A list of dictionaries representing valid transactions.
    """
    transactions = []

    for match in log_pattern.finditer(file_text_content):
        transaction = match.groupdict()
        transaction["price"] = float(transaction["price"])
        transactions.append(transaction)

    return transactions

In [15]:
transactions = parse_log_file(file_text_content)

print(f"Number of valid transactions: {len(transactions)}")
print(transactions[:3])

Number of valid transactions: 80
[{'timestamp': '2026-10-15 07:00:38', 'register_id': 'POS_01', 'item': 'Croissant', 'price': 3.98, 'currency': 'EUR', 'status': 'SUCCESS'}, {'timestamp': '2026-10-15 07:02:55', 'register_id': 'POS_01', 'item': 'Sandwich', 'price': 7.7, 'currency': 'USD', 'status': 'SUCCESS'}, {'timestamp': '2026-10-15 07:05:45', 'register_id': 'POS_01', 'item': 'Sandwich', 'price': 7.43, 'currency': 'USD', 'status': 'SUCCESS'}]
